# Generate and save Potts DMRG ground states

This is the **expensive** half of the entanglement-spectrum workflow. It runs DMRG and immediately saves every accepted TeNPy MPS in `data/ground_states/`. It contains no plotting.

There is one file per physical Hamiltonian. A converged state is preferred to an unconverged state, then lower variational energy is preferred; at equal energy, larger achieved bond dimension is preferred.

## 0. Imports

In [1]:
from pathlib import Path
import sys
import time

import numpy as np
import tenpy
from tenpy.algorithms import dmrg
from tenpy.networks.mps import MPS


def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "potts_model_project", cwd.parent, cwd.parent / "potts_model_project"]
    for candidate in candidates:
        if (candidate / "potts_model" / "potts.py").is_file():
            return candidate
    raise RuntimeError("Could not locate the potts_model_project folder.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from potts_model import PottsChain
from potts_model.ground_state_io import (
    ground_state_path,
    read_ground_state_metadata,
    save_ground_state,
    scan_ground_states,
)

GROUND_STATE_DIR = PROJECT_ROOT / "data" / "ground_states"
GROUND_STATE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Ground-state folder:", GROUND_STATE_DIR)
print("TeNPy version:", tenpy.__version__)

Project root: /Users/Aditya/Desktop/Third Year Summer Project/potts_model_project
Ground-state folder: /Users/Aditya/Desktop/Third Year Summer Project/potts_model_project/data/ground_states
TeNPy version: 1.1.0


## 1. Storage format

An MPS is stored as binary HDF5 rather than text. The human-readable filename includes $L$, $J$, $h$, open/periodic geometry, both boundary restrictions, and MPS ordering. The file metadata additionally records energy, requested and achieved bond dimension, sweeps, convergence, tolerances, runtime, canonical-form error, and software versions.

The metadata is duplicated in a lightweight HDF5 header, allowing the reader to identify files without loading their tensor data.

## 2. Numerical parameters

In [2]:

L_values =     [21, 22, 31, 32, 41, 42, 51, 52, 61, 62, 71, 72, 81, 82, 91, 92, 101, 102, 111, 112, 121, 122, 131, 132, 141, 142]
chi_max_values=[40, 40, 45, 45, 50, 50, 55, 55, 60, 60, 65, 65, 70, 70, 75, 75,  80,  80,  85,  85,  90,  90,  95,  95, 100, 100]
max_sweeps = 500

J = -1.0
h = -1.0
svd_min = 1e-11
max_E_err = 1e-9

n_levels_to_store = 40
force_rerun = False  # True intentionally repeats and replaces requested runs.
n_levels_to_plot = 24
reference_rank = 5       # sixth eigenvalue = fourth distinct Potts level group
reference_value = 2.0

if any(L < 4 for L in L_values):
    raise ValueError("Every entry in L_values must be an integer of at least 4.")
if len(chi_max_values) != len(L_values):
    raise ValueError("chi_max_values must have the same length as L_values.")
if any(chi <= 0 for chi in chi_max_values):
    raise ValueError("Every entry in chi_max_values must be positive.")

run_configs = list(zip(L_values, chi_max_values))
if len(set(run_configs)) != len(run_configs):
    raise ValueError("Each (L, chi_max) pair must be unique.")
if n_levels_to_store <= reference_rank:
    raise ValueError("n_levels_to_store must include the reference level.")

print("Runs (L, chi_max):", run_configs)
print("J = h =", J)
print("max_sweeps:", max_sweeps)

Runs (L, chi_max): [(21, 40), (22, 40), (31, 45), (32, 45), (41, 50), (42, 50), (51, 55), (52, 55), (61, 60), (62, 60), (71, 65), (72, 65), (81, 70), (82, 70), (91, 75), (92, 75), (101, 80), (102, 80), (111, 85), (112, 85), (121, 90), (122, 90), (131, 95), (132, 95), (141, 100), (142, 100)]
J = h = -1.0
max_sweeps: 500


## 3. Boundary-condition selection

The reusable `PottsChain` accepts `1`, `2`, or `3` for a fixed edge; `-1`, `-2`, or `-3` for an edge that forbids that state; and `"free"` for a free edge. Edit only `selected_case_labels` to compare a different subset.

For the periodic case, `order="default"` is intentional. The central MPS cut must divide the physical ring into the two contiguous half-chains used in the paper. A folded MPS order makes the Hamiltonian cheaper but changes which physical sites lie on each side of the central MPS cut.

In [3]:
all_boundary_cases = {
    "free|free": {"left": "free", "right": "free"},
    "1|1": {"left": 1, "right": 1},
    "1|2": {"left": 1, "right": 2},
    "1|3": {"left": 1, "right": 3},
    "1|-1": {"left": 1, "right": -1},
    "1|-2": {"left": 1, "right": -2},
    "1|free": {"left": 1, "right": "free"},
    "-1|-1": {"left": -1, "right": -1},
    "-1|-2": {"left": -1, "right": -2},
    "-1|free": {"left": -1, "right": "free"},
    "periodic": {
        "left": "free",
        "right": "free",
        "periodic": True,
        "order": "default",
    },
}

selected_case_labels = ["free|free", "1|1", "1|2", "-1|-1", "-1|-2"]

unknown_labels = set(selected_case_labels) - set(all_boundary_cases)
if unknown_labels:
    raise KeyError(f"Unknown boundary labels: {sorted(unknown_labels)}")

boundary_cases = {
    label: all_boundary_cases[label]
    for label in selected_case_labels
}

boundary_cases

{'free|free': {'left': 'free', 'right': 'free'},
 '1|1': {'left': 1, 'right': 1},
 '1|2': {'left': 1, 'right': 2},
 '-1|-1': {'left': -1, 'right': -1},
 '-1|-2': {'left': -1, 'right': -2}}

## 4. Central-cut spectrum checks

For even \(L\), the unique central bond is \(b=L/2\). For odd \(L=2m+1\), the two equally central bonds are \(b=m\) and \(b=m+1\), corresponding to partitions \(m|m+1\) and \(m+1|m\). Both odd-\(L\) central cuts are validated before the MPS is saved. These checks use Schmidt values already present in the state and do not add another optimization.


In [4]:
def central_cut_positions(L):
    """Return the unique central cut for even L or both central cuts for odd L."""
    if L < 2:
        raise ValueError("A bipartition requires L >= 2.")
    lower = L // 2
    return (lower,) if L % 2 == 0 else (lower, lower + 1)


def spectrum_at_cut(psi, cut):
    """Return all finite xi=-2 log(s) levels at MPS bond cut."""
    if not 1 <= cut < psi.L:
        raise ValueError(f"Cut {cut} is outside the internal bonds of an L={psi.L} MPS.")
    s = np.asarray(psi.get_SL(cut), dtype=float)
    s = s[np.isfinite(s) & (s > 0.0)]
    return np.sort(-2.0 * np.log(s))


def direct_tenpy_spectrum_at_cut(psi, cut):
    """Read TeNPy's xi=-log(lambda) spectrum for the same MPS bond."""
    xi = np.asarray(psi.entanglement_spectrum()[cut - 1], dtype=float)
    return np.sort(xi[np.isfinite(xi)])


def normalize_spectrum(xi, rank, target=reference_value):
    """Shift xi[0] to zero and set xi[rank] to target."""
    xi = np.sort(np.asarray(xi, dtype=float))
    if len(xi) <= rank:
        raise ValueError(f"Need at least {rank + 1} levels for this normalization.")
    gap = xi[rank] - xi[0]
    if not np.isfinite(gap) or gap <= 1e-12:
        raise ValueError("The selected reference gap is zero or numerically unstable.")
    return target * (xi - xi[0]) / gap


def validate_spectrum_at_cut(psi, cut, atol=2e-10):
    """Validate normalization and entropy reconstruction at one specified cut."""
    xi = spectrum_at_cut(psi, cut)
    xi_direct = direct_tenpy_spectrum_at_cut(psi, cut)
    if len(xi) != len(xi_direct) or not np.allclose(
        xi, xi_direct, atol=atol, rtol=atol
    ):
        raise RuntimeError(
            f"TeNPy's direct spectrum disagrees with -2 log(Schmidt values) at cut {cut}."
        )

    lambdas = np.exp(-xi)
    lambda_sum = float(np.sum(lambdas))
    entropy_from_spectrum = float(np.sum(lambdas * xi))
    entropy_from_tenpy = float(psi.entanglement_entropy()[cut - 1])

    if not np.isclose(lambda_sum, 1.0, atol=atol, rtol=atol):
        raise RuntimeError(
            f"Density-matrix eigenvalues at cut {cut} sum to {lambda_sum}, not 1."
        )
    if not np.isclose(entropy_from_spectrum, entropy_from_tenpy, atol=atol, rtol=atol):
        raise RuntimeError(f"The spectrum does not reconstruct the entropy at cut {cut}.")

    return {
        "cut": int(cut),
        "partition": [int(cut), int(psi.L - cut)],
        "cut_parity": "even" if cut % 2 == 0 else "odd",
        "lambda_sum": lambda_sum,
        "entropy_from_spectrum": entropy_from_spectrum,
        "entropy_from_tenpy": entropy_from_tenpy,
    }


def validate_central_spectra(psi):
    """Validate every cut adjacent to the geometric centre."""
    cuts = central_cut_positions(psi.L)
    by_cut = {str(cut): validate_spectrum_at_cut(psi, cut) for cut in cuts}
    return {"central_cuts": [int(cut) for cut in cuts], "by_cut": by_cut}


## 5. Model builder, DMRG runner, and save metadata

The physical Hamiltonian parameters determine the filename. Numerical settings describe the accuracy of the state stored inside that file.

In [5]:
def build_model(L, case):
    return PottsChain.from_parameters(
        L=L,
        J=J,
        h=h,
        left_boundary=case["left"],
        right_boundary=case["right"],
        periodic=case.get("periodic", False),
        order=case.get("order", "default"),
    )


def antiferromagnetic_initial_state(model):
    """Alternating product state compatible with every restricted site."""
    physical_state = []
    L = model.lat.N_sites

    for site_index, allowed in enumerate(model.allowed_states_by_site):
        preferred = 1 if site_index % 2 == 0 else 2
        candidates = [preferred, 1, 2, 3]
        forbidden = {physical_state[-1]} if physical_state else set()

        # Look ahead to an exactly fixed right edge, so a third colour can
        # avoid an unnecessary equal-state bond (for example, equal fixed edges).
        if site_index == L - 2:
            right_allowed = model.allowed_states_by_site[-1]
            if len(right_allowed) == 1:
                forbidden.add(right_allowed[0])

        # On a ring, also make the final site differ from the first one.
        if model.is_periodic and site_index == L - 1:
            forbidden.add(physical_state[0])

        selected = next(
            (state for state in candidates if state in allowed and state not in forbidden),
            allowed[0],
        )
        physical_state.append(selected)

    return [str(physical_state[index]) for index in model._mps_to_physical]


def run_dmrg_for_case(L, chi_for_L, label, case):
    model = build_model(L, case)
    if J > 0:
        initial_state = model.initial_product_state(bulk_state=1)
    else:
        initial_state = antiferromagnetic_initial_state(model)
    psi = MPS.from_product_state(
        model.lat.mps_sites(),
        initial_state,
        bc=model.lat.bc_MPS,
        unit_cell_width=model.lat.mps_unit_cell_width,
    )

    dmrg_params = {
        "mixer": True,
        "max_sweeps": max_sweeps,
        "N_sweeps_check": 1,
        "max_E_err": max_E_err,
        "max_N_sites_per_ring": model.lat.N_sites,
        "trunc_params": {
            "chi_max": chi_for_L,
            "svd_min": svd_min,
        },
    }

    engine = dmrg.TwoSiteDMRGEngine(psi, model, dmrg_params)
    start_time = time.perf_counter()
    energy, psi = engine.run()
    elapsed_seconds = time.perf_counter() - start_time

    if engine.shelve:
        final_status = "stopped at time limit"
    elif engine.is_converged():
        final_status = "converged"
    else:
        final_status = "sweep limit reached"
    central_checks = validate_central_spectra(psi)
    reference_cut = L // 2
    xi_all = spectrum_at_cut(psi, reference_cut)
    reference_checks = central_checks["by_cut"][str(reference_cut)]
    checks = {
        **reference_checks,
        "reference_cut": int(reference_cut),
        "central_cuts": central_checks["central_cuts"],
        "by_cut": central_checks["by_cut"],
    }
    xi = xi_all[:n_levels_to_store]

    return {
        "label": label,
        "case": case,
        "L": L,
        "chi_max_requested": chi_for_L,
        "model": model,
        "psi": psi,
        "energy": float(np.real(energy)),
        "energy_per_site": float(np.real(energy) / L),
        "xi_raw": xi,
        "xi_shifted": xi - xi[0],
        "x_inv_log_L": 1.0 / np.log(L),
        "checks": checks,
        "chi": list(psi.chi),
        "sweeps": engine.sweeps,
        "status": final_status,
        "elapsed_seconds": elapsed_seconds,
    }


def physics_metadata(L, label, case):
    return {
        "model": "three_state_quantum_potts",
        "hamiltonian_convention": "PottsChain Equation-7 projector convention",
        "L": int(L),
        "J": float(J),
        "h": float(h),
        "boundary_label": label,
        "left_boundary": case["left"],
        "right_boundary": case["right"],
        "periodic": bool(case.get("periodic", False)),
        "order": case.get("order", "default"),
        "bc_MPS": "finite",
    }


def path_for_run(L, label, case):
    return ground_state_path(
        {"physics": physics_metadata(L, label, case), "numerics": {}},
        GROUND_STATE_DIR,
    )


def metadata_for_result(result):
    canonical_norm_error = float(np.linalg.norm(result["psi"].norm_test()))
    return {
        "physics": physics_metadata(result["L"], result["label"], result["case"]),
        "numerics": {
            "energy": result["energy"],
            "energy_per_site": result["energy_per_site"],
            "status": result["status"],
            "converged": result["status"] == "converged",
            "chi_max_requested": result["chi_max_requested"],
            "chi_max_achieved": max(result["chi"]),
            "chi_profile": result["chi"],
            "sweeps": result["sweeps"],
            "elapsed_seconds": result["elapsed_seconds"],
            "max_sweeps": max_sweeps,
            "max_E_err": max_E_err,
            "svd_min": svd_min,
            "checks": {**result["checks"], "canonical_norm_error": canonical_norm_error},
        },
        "software": {
            "tenpy_version": tenpy.__version__,
            "python_version": sys.version.split()[0],
        },
    }


## 6. Run and save

This is the expensive cell. Each state is saved immediately after its run. A stored converged state with at least the requested bond-dimension cap is skipped before DMRG. Set `force_rerun=True` only when you deliberately want to repeat every requested calculation.

In [6]:
for label, case in boundary_cases.items():
    for L, chi_for_L in run_configs:
        destination = path_for_run(L, label, case)

        if destination.exists() and not force_rerun:
            stored = read_ground_state_metadata(destination)
            stored_numerics = stored["numerics"]
            sufficient = (
                stored_numerics.get("status") == "converged"
                and int(stored_numerics.get("chi_max_requested", 0)) >= chi_for_L
            )
            if sufficient:
                print(
                    f"Skipping {label:>10}, L={L:>3}, chi_max={chi_for_L}: "
                    f"stored converged state has requested chi="
                    f"{stored_numerics['chi_max_requested']}."
                )
                continue

        print(f"Running {label:>10}, L={L:>3}, chi_max={chi_for_L} ...", flush=True)
        result = run_dmrg_for_case(L, chi_for_L, label, case)
        decision = save_ground_state(
            result["psi"],
            metadata_for_result(result),
            GROUND_STATE_DIR,
            force=force_rerun,
        )
        canonical_error = float(np.linalg.norm(result["psi"].norm_test()))
        print(
            f"  {decision['action']}: {decision['path'].name}\n"
            f"  {result['status']} after {result['sweeps']} sweeps "
            f"({result['elapsed_seconds']:,.1f} s); E/L={result['energy_per_site']:.10f}; "
            f"max chi={max(result['chi'])}; canonical error={canonical_error:.3e}\n"
            f"  Decision: {decision['reason']}"
        )

print("Finished all requested runs.")

Running  free|free, L= 21, chi_max=40 ...
  saved: potts_L0021_Jm1_hm1_open_Lfree_Rfree_default.h5
  converged after 6 sweeps (2.1 s); E/L=-0.5941053257; max chi=40; canonical error=1.975e-14
  Decision: no stored state existed
Running  free|free, L= 22, chi_max=40 ...
  saved: potts_L0022_Jm1_hm1_open_Lfree_Rfree_default.h5
  converged after 6 sweeps (2.3 s); E/L=-0.5948511477; max chi=40; canonical error=1.585e-14
  Decision: no stored state existed
Running  free|free, L= 31, chi_max=45 ...
  saved: potts_L0031_Jm1_hm1_open_Lfree_Rfree_default.h5
  converged after 11 sweeps (6.6 s); E/L=-0.5977619845; max chi=45; canonical error=2.586e-14
  Decision: no stored state existed
Running  free|free, L= 32, chi_max=45 ...
  saved: potts_L0032_Jm1_hm1_open_Lfree_Rfree_default.h5
  converged after 10 sweeps (6.6 s); E/L=-0.5981100753; max chi=45; canonical error=2.544e-14
  Decision: no stored state existed
Running  free|free, L= 41, chi_max=50 ...
  saved: potts_L0041_Jm1_hm1_open_Lfree_Rfre

final DMRG state not in canonical form up to norm_tol=1.00e-05: norm_err=1.04e-05


  saved: potts_L0102_Jm1_hm1_open_Lfix1_Rfix1_default.h5
  converged after 15 sweeps (83.0 s); E/L=-0.5970392164; max chi=80; canonical error=3.660e-14
  Decision: no stored state existed
Running        1|1, L=111, chi_max=85 ...
  saved: potts_L0111_Jm1_hm1_open_Lfix1_Rfix1_default.h5
  converged after 8 sweeps (53.7 s); E/L=-0.5978032193; max chi=85; canonical error=5.371e-14
  Decision: no stored state existed
Running        1|1, L=112, chi_max=85 ...
  saved: potts_L0112_Jm1_hm1_open_Lfix1_Rfix1_default.h5
  converged after 13 sweeps (110.9 s); E/L=-0.5977894019; max chi=85; canonical error=4.171e-14
  Decision: no stored state existed
Running        1|1, L=121, chi_max=90 ...
  saved: potts_L0121_Jm1_hm1_open_Lfix1_Rfix1_default.h5
  converged after 7 sweeps (63.5 s); E/L=-0.5984271682; max chi=90; canonical error=5.262e-14
  Decision: no stored state existed
Running        1|1, L=122, chi_max=90 ...


final DMRG state not in canonical form up to norm_tol=1.00e-05: norm_err=1.08e-05


  saved: potts_L0122_Jm1_hm1_open_Lfix1_Rfix1_default.h5
  converged after 15 sweeps (139.9 s); E/L=-0.5984154787; max chi=90; canonical error=4.192e-14
  Decision: no stored state existed
Running        1|1, L=131, chi_max=95 ...
  saved: potts_L0131_Jm1_hm1_open_Lfix1_Rfix1_default.h5
  converged after 7 sweeps (76.4 s); E/L=-0.5989559075; max chi=95; canonical error=5.585e-14
  Decision: no stored state existed
Running        1|1, L=132, chi_max=95 ...
  saved: potts_L0132_Jm1_hm1_open_Lfix1_Rfix1_default.h5
  converged after 16 sweeps (185.4 s); E/L=-0.5989458901; max chi=95; canonical error=4.340e-14
  Decision: no stored state existed
Running        1|1, L=141, chi_max=100 ...
  saved: potts_L0141_Jm1_hm1_open_Lfix1_Rfix1_default.h5
  converged after 7 sweeps (94.3 s); E/L=-0.5994096847; max chi=100; canonical error=6.915e-14
  Decision: no stored state existed
Running        1|1, L=142, chi_max=100 ...


...

  saved: potts_L0122_Jm1_hm1_open_Lforbid1_Rforbid1_default.h5
  converged after 17 sweeps (154.0 s); E/L=-0.6027823349; max chi=90; canonical error=4.054e-14
  Decision: no stored state existed
Running      -1|-1, L=131, chi_max=95 ...
  saved: potts_L0131_Jm1_hm1_open_Lforbid1_Rforbid1_default.h5
  converged after 7 sweeps (77.8 s); E/L=-0.6030222340; max chi=95; canonical error=6.005e-14
  Decision: no stored state existed
Running      -1|-1, L=132, chi_max=95 ...


final DMRG state not in canonical form up to norm_tol=1.00e-05: norm_err=1.30e-05


  saved: potts_L0132_Jm1_hm1_open_Lforbid1_Rforbid1_default.h5
  converged after 15 sweeps (186.7 s); E/L=-0.6029818529; max chi=95; canonical error=4.204e-14
  Decision: no stored state existed
Running      -1|-1, L=141, chi_max=100 ...
  saved: potts_L0141_Jm1_hm1_open_Lforbid1_Rforbid1_default.h5
  converged after 7 sweeps (95.1 s); E/L=-0.6031876225; max chi=100; canonical error=5.490e-14
  Decision: no stored state existed
Running      -1|-1, L=142, chi_max=100 ...


final DMRG state not in canonical form up to norm_tol=1.00e-05: norm_err=1.25e-05


  saved: potts_L0142_Jm1_hm1_open_Lforbid1_Rforbid1_default.h5
  converged after 18 sweeps (251.9 s); E/L=-0.6031526925; max chi=100; canonical error=4.445e-14
  Decision: no stored state existed
Running      -1|-2, L= 21, chi_max=40 ...
  saved: potts_L0021_Jm1_hm1_open_Lforbid1_Rforbid2_default.h5
  converged after 5 sweeps (1.7 s); E/L=-0.5898917266; max chi=40; canonical error=1.898e-14
  Decision: no stored state existed
Running      -1|-2, L= 22, chi_max=40 ...
  saved: potts_L0022_Jm1_hm1_open_Lforbid1_Rforbid2_default.h5
  converged after 4 sweeps (1.6 s); E/L=-0.5913191420; max chi=40; canonical error=1.961e-14
  Decision: no stored state existed
Running      -1|-2, L= 31, chi_max=45 ...
  saved: potts_L0031_Jm1_hm1_open_Lforbid1_Rforbid2_default.h5
  converged after 5 sweeps (3.5 s); E/L=-0.5950696508; max chi=45; canonical error=1.785e-14
  Decision: no stored state existed
Running      -1|-2, L= 32, chi_max=45 ...
  saved: potts_L0032_Jm1_hm1_open_Lforbid1_Rforbid2_default.

## 7. Saved-state inventory

This reads only metadata headers and does not load MPS tensors.

In [7]:
records = scan_ground_states(GROUND_STATE_DIR)
valid_records = [record for record in records if "index_error" not in record]
invalid_records = [record for record in records if "index_error" in record]

print(
    f"{'file':<72} {'status':<20} {'chi':>5} {'sweeps':>7} "
    f"{'E/L':>15} {'canonical err':>14}"
)
print("-" * 140)
for record in valid_records:
    numerics = record["numerics"]
    checks = numerics.get("checks", {})
    print(
        f"{record['filename']:<72} {numerics.get('status', '?'):<20} "
        f"{int(numerics.get('chi_max_achieved', 0)):5d} "
        f"{int(numerics.get('sweeps', 0)):7d} "
        f"{float(numerics.get('energy_per_site', np.nan)):15.10f} "
        f"{float(checks.get('canonical_norm_error', np.nan)):14.3e}"
    )

if invalid_records:
    print("\nFiles that could not be indexed:")
    for record in invalid_records:
        print(" ", record["filename"], "->", record["index_error"])

file                                                                     status                 chi  sweeps             E/L  canonical err
--------------------------------------------------------------------------------------------------------------------------------------------
potts_L0020_Jm1_hm1_open_Lfix1_Rfix1_default.h5                          converged               50       7   -0.5610246870      1.779e-14
potts_L0020_Jm1_hm1_open_Lfix1_Rfix2_default.h5                          converged               50       4   -0.5632613420      1.274e-14
potts_L0020_Jm1_hm1_open_Lforbid1_Rforbid1_default.h5                    converged               50       6   -0.5877751165      1.488e-14
potts_L0020_Jm1_hm1_open_Lforbid1_Rforbid2_default.h5                    converged               50       4   -0.5899030381      1.583e-14
potts_L0020_Jm1_hm1_open_Lfree_Rfree_default.h5                          converged               50       5   -0.5938128318      1.935e-14
potts_L0021_Jm1_hm1_open_